Creating corpus

In [2]:
import pandas as pd
from nltk.tokenize import word_tokenize
import string
import json

In [3]:
df_details = pd.read_csv("dw_data/all-detailsepisodes.csv") # columns: episodeid, title, first_diffusion, doctorid
df_imdb = pd.read_csv("dw_data/imdb_details.csv") # columns: number, title, rating, nbr_votes, description, season
# Merge datasets on title
df = pd.merge(df_details, df_imdb, on="title", how="inner")
df.to_csv("dw_data/merged_dataset.csv", index=False)

In [4]:
def preprocess(text):
    if pd.isna(text):
        return []
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in string.punctuation]
    return tokens

In [5]:
from collections import defaultdict

inverted_index = defaultdict(list)
document_corpus = {}

# Preparing the corpus for building the inverted index
descriptions = df['description'].fillna("").tolist()
episode_keys = (df['season'].astype(str) + "x" + df['number'].astype(str)).tolist()
titles = df['title'].tolist()

# Filling the index
for i, doc in enumerate(descriptions):
    doc_id = episode_keys[i] # unique identifier for each episode, e.g., "1x01"
    
    document_corpus[doc_id] = {
        "title": titles[i],
        "description": doc,
        "id": doc_id
    }
    tokens = preprocess(doc) 
    
    for token in tokens:
        if doc_id not in inverted_index[token]:
            inverted_index[token].append(doc_id)

print(f"Index built with {len(inverted_index)} unique terms for {len(document_corpus)} episodes.")

Index built with 1512 unique terms for 135 episodes.


In [7]:
# Сортировка и фильтрация без загрузки из JSON
def sort_key(epid):
    season, number = epid.split('x')
    return int(season), int(number)

sorted_corpus = dict(sorted(document_corpus.items(), key=lambda x: sort_key(x[0])))
filtered_corpus = {k: v for k, v in sorted_corpus.items() if not k.startswith("11x")}

with open("dw_data/document_corpus_dw.json", "w", encoding="utf-8") as f:
    json.dump(filtered_corpus, f, ensure_ascii=False, indent=2)

with open("dw_data/inverted_index.json", "w", encoding="utf-8") as f:
    json.dump(dict(inverted_index), f, ensure_ascii=False, indent=2)

print(f"Filtered JSON saved. {len(document_corpus) - len(filtered_corpus)} episodes removed.")

Filtered JSON saved. 10 episodes removed.
